In [4]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & SECURE API AUTHENTICATION
# Run this cell first. Installs the Google GenAI SDK, Pydantic, and dotenv.
# ==============================================================================
!pip install -q -U google-genai pydantic python-dotenv tabulate

import os
import sys
import time
import json
import random
import getpass
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

# Official Google GenAI SDK
from google import genai
from google.genai import types
from google.genai.errors import APIError

# ------------------------------------------------------------------------------
# Secure Gemini API Key Ingestion (Zero-Hardcoding Policy)
# ------------------------------------------------------------------------------
# Never hardcode API keys directly in scripts!
# In Google Colab, use the Secrets Manager (🔑 icon on the left panel) as 'GEMINI_API_KEY'.
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("🔑 Enter your Google Gemini API Key: ")
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# Initialize Client
client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Google Gemini API Client initialized successfully!")

✅ Google Gemini API Client initialized successfully!


In [5]:
# ==============================================================================
# SECTION 1: API ARCHITECTURE, STATELESSNESS & SECURITY HYGIENE
# ==============================================================================
"""
1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:
   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.
   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.
   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.

2. THE STATELESSNESS MENTAL MODEL:
   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.
   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list
     of previous (User, Model) turns and pass the cumulative array on every subsequent call.

3. MESSAGE ROLES MAPPING:
   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐
   │ Role Type            │ OpenAI / Anthropic      │ Google Gemini API         │
   ├──────────────────────┼─────────────────────────┼───────────────────────────┤
   │ System Persona/Rules │ role: 'system'          │ config.system_instruction │
   │ User Message         │ role: 'user'            │ role: 'user'              │
   │ Model Response       │ role: 'assistant'       │ role: 'model'             │
   └──────────────────────┴─────────────────────────┴───────────────────────────┘
"""

"\n1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:\n   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.\n   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.\n   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.\n\n2. THE STATELESSNESS MENTAL MODEL:\n   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.\n   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list\n     of previous (User, Model) turns and pass the cumulative array on every subsequent call.\n\n3. MESSAGE ROLES MAPPING:\n   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐\n   │ Role Type            │ OpenAI / Anthropic      │ Google Gemini API         │\n   ├──────────────────────┼─────────────────────────┼───────────────────────────┤\n   │ System Persona/Rules │ role: 'system'          │ config.system_instruction │\

In [6]:
# ==============================================================================
# SECTION 2: PRE-FLIGHT TOKEN COUNTING & FINANCIAL COST ESTIMATION (UPDATED)
# ==============================================================================
"""
COST ESTIMATION BEST PRACTICE:
Count input tokens BEFORE invoking expensive generation calls to protect budget thresholds.
"""
import numpy as np
def preflight_cost_estimate(
    text_prompt: str,
    model_name: str = "gemini-3.6-flash",
    expected_output_tokens: int = 500
) -> Dict[str, Any]:
    """Calculates exact input tokens and estimates financial cost before calling the API."""
    # Count tokens using official Gemini Tokenizer
    token_resp = client.models.count_tokens(model=model_name, contents=text_prompt)
    input_tokens = token_resp.total_tokens

    # Official Rates per 1M tokens (USD)
    pricing = {
        "gemini-3.6-flash": {"in": 0.075, "out": 0.30},
        "gemini-1.5-pro":   {"in": 1.25,  "out": 5.00}
    }
    rate = pricing.get(model_name, pricing["gemini-3.6-flash"])

    est_cost = (input_tokens / 1e6 * rate["in"]) + (expected_output_tokens / 1e6 * rate["out"])

    return {
        "model": model_name,
        "input_tokens": input_tokens,
        "estimated_output_tokens": expected_output_tokens,
        "estimated_cost_usd": np.round(est_cost, 6),
        "cost_per_10k_calls": np.round(est_cost * 10000, 2)
    }

sample_payload = "Please summarize the last 10 quarterly financial filings of Apple, Microsoft, and Google."
# Updated to gemini-3.6-flash
estimate = preflight_cost_estimate(sample_payload, model_name="gemini-3.6-flash")
print("=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===")
for k, v in estimate.items():
    print(f"• {k:25s}: {v}")

=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===
• model                    : gemini-3.6-flash
• input_tokens             : 19
• estimated_output_tokens  : 500
• estimated_cost_usd       : 0.000151
• cost_per_10k_calls       : 1.51


In [7]:
# ==============================================================================
# SECTION 3: PRODUCTION RESILIENCE — EXPONENTIAL BACKOFF & RETRY LOOP
# ==============================================================================
"""
HANDLING API FAILURES IN PRODUCTION:
1. Rate Limits (HTTP 429 / ResourceExhausted): Hit requests-per-minute (RPM) ceiling.
2. Transient Server Errors (HTTP 500 / 503): Temporary Google Cloud infrastructure hiccup.
3. Network Timeouts: Connection dropped during streaming.

REMEDY: EXPONENTIAL BACKOFF WITH JITTER:
Wait time = (base_delay * 2^attempt) + random_jitter
Prevents "Thundering Herd" problem where all failed clients retry at the exact same millisecond.
"""

def execute_with_exponential_backoff(
    api_call_func,
    max_retries: int = 4,
    base_delay: float = 1.5
):
    """Wraps an API call in an exponential backoff retry loop with random jitter."""
    for attempt in range(max_retries):
        try:
            return api_call_func()
        except APIError as e:
            if attempt == max_retries - 1:
                print(f"❌ Max retries reached. Fatal API Error: {e}")
                raise e
            # Calculate backoff delay with jitter
            delay = (base_delay * (2 ** attempt)) + random.uniform(0.1, 0.8)
            print(f"⚠️ Warning: Transient API Error ({e.code}). Retrying in {delay:.2f}s... (Attempt {attempt+1}/{max_retries})")
            time.sleep(delay)

In [8]:
# ==============================================================================
# SECTION 4: REUSABLE GEMINI WRAPPER & 3-TURN CHAT
# ==============================================================================
# Construct a reusable production function supporting:
# - Streaming responses (Low Time-To-First-Token)
# - System instructions
# - Dynamic temperature
# - Exponential backoff retry logic

def gemini_call(
    prompt: str,
    system_instruction: str = "You are a concise, helpful enterprise AI assistant.",
    temperature: float = 0.2,
    stream: bool = False,
    model: str = "gemini-3.6-flash"
) -> str:
    """Production-grade wrapper for Google Gemini API with error handling and streaming."""
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        max_output_tokens=800
    )

    if stream:
        def stream_call():
            full_text = []
            response_stream = client.models.generate_content_stream(
                model=model, contents=prompt, config=config
            )
            for chunk in response_stream:
                if chunk.text:
                    print(chunk.text, end="", flush=True)
                    full_text.append(chunk.text)
            print() # Print final newline
            return "".join(full_text)

        return execute_with_exponential_backoff(stream_call)
    else:
        def standard_call():
            resp = client.models.generate_content(
                model=model, contents=prompt, config=config
            )
            return resp.text.strip()

        return execute_with_exponential_backoff(standard_call)

# ------------------------------------------------------------------------------
# 3-Turn Conversational Memory Loop Demonstration
# ------------------------------------------------------------------------------
print("=== MULTI-TURN CONVERSATION LOOP ===")

# Explicitly maintain stateless conversation history
conversation_history = []
system_persona = "You are a Senior PostgreSQL Database Administrator. Answer concisely in 2 sentences."

def send_chat_turn(user_message: str):
    print(f"\n👤 User: {user_message}")
    print("🤖 Assistant: ", end="")

    # 1. Append user message to history
    conversation_history.append({"role": "user", "parts": [{"text": user_message}]})

    # 2. Call Gemini passing full conversation history
    config = types.GenerateContentConfig(
        system_instruction=system_persona,
        temperature=0.0
    )
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=conversation_history,
        config=config
    )

    bot_reply = response.text.strip()
    print(bot_reply)

    # 3. Append model response to history to maintain context
    conversation_history.append({"role": "model", "parts": [{"text": bot_reply}]})

# Execute 3-Turn Dialogue (Demonstrating Context Memory)
send_chat_turn("What is the difference between a clustered and non-clustered index?")
send_chat_turn("Which one is faster for range queries on primary keys?") # Pronoun resolution!
send_chat_turn("Can a table have multiple of the faster one?")           # Contextual follow-up!

=== MULTI-TURN CONVERSATION LOOP ===

👤 User: What is the difference between a clustered and non-clustered index?
🤖 Assistant: A clustered index physically reorders the actual table data on disk to match the index order, meaning a table can only have one. In contrast, a non-clustered index maintains a separate structure containing search keys and pointers to the physical data rows, allowing multiple non-clustered indexes per table.

👤 User: Which one is faster for range queries on primary keys?
🤖 Assistant: A clustered index is significantly faster for range queries because the data rows are physically stored in sequential order on disk, allowing the database to perform efficient sequential reads. In contrast, a non-clustered index requires retrieving rows from non-contiguous pages across the disk, resulting in slower random I/O operations for every record in the range.

👤 User: Can a table have multiple of the faster one?
🤖 Assistant: No, a table can only have one clustered index beca

In [ ]:
# ==============================================================================
# SECTION 5: STUDENT LAB WORKSPACE (PORTFOLIO APPLICATION)
# ==============================================================================
"""
🎓 STUDENT LAB ASSIGNMENT:
Build an end-to-end AI Application: "The Executive Resume Bullet & Impact Optimizer"

APPLICATION REQUIREMENTS:
1. Structured JSON Schema (Pydantic):
   - `original_bullet`: Raw user text
   - `xyz_formatted_bullet`: Rewritten using Google's XYZ Formula:
     "Accomplished [X], as measured by [Y], by doing [Z]"
   - `impact_metric`: The quantifiable numeric KPI
   - `action_verb`: Strong opening action verb
   - `seniority_score`: Integer rating (1 to 10) of executive presence
   - `critique`: 1-sentence explanation of what was improved
2. Interactive Revision History: Allow user to request a revision (multi-turn).
3. Streaming or Schema Parsing: Correctly parse and display output.
4. Error Handling: Enclose calls in retry blocks.
"""

# ==============================================================================
# TASK 1: DEFINE PYDANTIC SCHEMA FOR STRUCTURED RESUME OPTIMIZATION
# ==============================================================================

class OptimizedBullet(BaseModel):
    original_bullet: str = Field(description="The raw, original resume bullet point provided by the user.")
    xyz_formatted_bullet: str = Field(description="The rewritten resume bullet point using the XYZ formula: 'Accomplished [X], as measured by [Y], by doing [Z]'.")
    impact_metric: str = Field(description="The quantifiable numeric KPI or result achieved.")
    action_verb: str = Field(description="A strong, impactful verb used to start the optimized bullet point.")
    seniority_score: int = Field(ge=1, le=10, description="An integer score from 1 to 10 rating the executive presence and strategic value of the bullet point, where 10 is highest.")
    critique: str = Field(description="A concise, 1-sentence explanation of what was improved in the bullet point.")

# ==============================================================================
# TASK 2: BUILD THE APPLICATION ENGINE
# ==============================================================================

def optimize_bullet_point(original_bullet: str) -> OptimizedBullet:
    """
    Optimizes a resume bullet point using the Gemini API, ensuring the output
    conforms to the OptimizedBullet Pydantic schema.
    """
    system_instruction = f"""
    You are an expert Executive Resume Writer. Your task is to transform a raw resume bullet point
    into an impactful, quantifiable statement using the XYZ formula:
    "Accomplished [X], as measured by [Y], by doing [Z]".
    Strictly adhere to the provided JSON schema for OptimizedBullet.

    Your focus areas for optimization are:
    1.  **Action Verb (X):** Start with a strong, active verb.
    2.  **Quantifiable Impact (Y):** Include specific metrics, numbers, or percentages to demonstrate achievement. If not present, infer or suggest, making it clear in the critique.
    3.  **Method/Skill (Z):** Briefly explain *how* the accomplishment was achieved, highlighting relevant skills.
    4.  **Executive Presence:** Assign a seniority score (1-10) reflecting strategic value and leadership.
    5.  **Critique:** Provide a single sentence explaining the key improvement made.
    """

    config = types.GenerateContentConfig(
        temperature=0.1, # Keep temperature low for consistent, structured output
        response_mime_type='application/json',
        response_schema=OptimizedBullet.model_json_schema(), # Use the Pydantic schema for output
        system_instruction=system_instruction
    )

    prompt = f"Optimize the following resume bullet point: '{original_bullet}'"

    def api_call():
        resp = client.models.generate_content(
            model="gemini-1.5-pro", # Using 1.5 Pro for better instruction following and JSON generation
            contents=prompt,
            config=config
        )
        return resp.text.strip() # The response is a JSON string

    json_response = execute_with_exponential_backoff(api_call)

    # Parse the JSON response into our Pydantic model
    try:
        optimized_data = json.loads(json_response)
        return OptimizedBullet(**optimized_data)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON response: {e}")
        print(f"Raw response: {json_response}")
        raise
    except Exception as e:
        print(f"Error parsing OptimizedBullet: {e}")
        print(f"Raw response: {json_response}")
        raise

# ==============================================================================
# TASK 3: TEST APPLICATION ON REAL-WORLD WEAK BULLETS
# ==============================================================================

# Example weak bullet points for testing
weak_bullets = [
    "Managed projects.",
    "Responsible for sales numbers.",
    "Helped customers.",
    "Did reports every week.",
    "Oversaw a team."
]

print("\n=== Executive Resume Bullet Optimizer Chatbot ===")
print("Enter a resume bullet point to optimize, or type 'quit' to exit.")

while True:
    user_input = input("\n👤 Your Bullet Point: ").strip()

    if user_input.lower() == 'quit':
        print("Exiting Chatbot. Goodbye!")
        break

    if not user_input:
        print("Please enter a bullet point to optimize.")
        continue

    try:
        print("🤖 Optimizing...")
        optimized_bullet = optimize_bullet_point(user_input)

        print("\n--- Optimized Bullet Point --- ")
        print(f"Original: {optimized_bullet.original_bullet}")
        print(f"XYZ Format: {optimized_bullet.xyz_formatted_bullet}")
        print(f"Action Verb: {optimized_bullet.action_verb}")
        print(f"Impact Metric: {optimized_bullet.impact_metric}")
        print(f"Seniority Score: {optimized_bullet.seniority_score}/10")
        print(f"Critique: {optimized_bullet.critique}")

    except Exception as e:
        print(f"An error occurred during optimization: {e}")
        print("Please try again or check your API key and network connection.")



=== Executive Resume Bullet Optimizer Chatbot ===
Enter a resume bullet point to optimize, or type 'quit' to exit.

👤 Your Bullet Point: 5
🤖 Optimizing...
⚠️ Warning: Transient API Error (404). Retrying in 1.79s... (Attempt 1/4)
⚠️ Warning: Transient API Error (404). Retrying in 3.78s... (Attempt 2/4)
⚠️ Warning: Transient API Error (404). Retrying in 6.64s... (Attempt 3/4)
❌ Max retries reached. Fatal API Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-pro is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
An error occurred during optimization: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-pro is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
P

In [10]:
# ==============================================================================
# SECTION 6: GIT REPOSITORY HYGIENE — CREATING .ENV AND .GITIGNORE
# ==============================================================================
"""
INSTRUCTIONS FOR PUSHING TO GITHUB SAFELY:

1. Create a `.env` file locally:
   GEMINI_API_KEY=your_actual_key_here

2. Create a `.gitignore` file in your project root containing:
   .env
   .env.local
   *.joblib
   __pycache__/
   .ipynb_checkpoints/

3. In your Python script (`app.py`), load the key cleanly via:
   from dotenv import load_dotenv
   load_dotenv()
   api_key = os.getenv("GEMINI_API_KEY")
"""

# Script to generate .gitignore locally in Colab
with open(".gitignore", "w") as f:
    f.write(".env\n.env.*\n*.joblib\n__pycache__/\n.ipynb_checkpoints/\n")

print("✅ '.gitignore' template created successfully!")

✅ '.gitignore' template created successfully!


In [11]:
#AQ.Ab8RN6JohcWtZf_5jWfO1e1qB3VuqIVyl9PjjzciGvGu_yNGHg